# I0 · Frozen DINOv2 ViT-B/14 + P3 + E4
Self-contained Colab T4 notebook. Only the backbone size and E4 input width change.
Inspected remote `baseline.ipynb` (F0 recovery) at repository commit `9464fdfb6075d8908fa05b95a3654671f1fea55e`.
Its P3/E4, train-only standardization and accuracy-based early stopping are reused here.
H2 exists but its documentation reports no executed fine-tuning results. The reported
F0 98% / 0.9803 / Kaggle 0.94693 are **user-provided historical references**, not reverified results.
Existing notebooks, F0 caches/checkpoints and submissions are never modified.


## 0. Environment and dependencies
Select Runtime → Change runtime type → T4 GPU. Keep Colab’s compatible torch/torchvision pair. Model download requires public GitHub and dl.fbaipublicfiles.com access; local extracted data requires no API keys.

In [ ]:
%pip -q install scikit-learn pandas pillow tqdm matplotlib


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from datetime import datetime, timezone
import uuid, random, hashlib, json, platform, time, warnings
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from IPython.display import display
print('Python', platform.python_version(), 'torch', torch.__version__, 'torchvision', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable; CPU is slow')


## 1. Configuration and reproducibility
Provide extracted train/val/test folders and original sample_submission.csv. Optional Drive persists caches and outputs. Default runs only head seed 2026. Explicitly set HEAD_SEEDS=[123,42,2026] for all three, sharing fixed augmentation-seed-123 embeddings. Each execution creates a fresh run directory; cache files live separately.

In [ ]:
USE_DRIVE = False
DATA_ROOT = Path('/content/car_data')
STORAGE_ROOT = Path('/content')  # Or /content/drive/MyDrive/Car-Classification-Benchmark
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/Car-Classification-Benchmark')
MODEL_NAME = 'dinov2_vitb14'
HUB_REF = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'  # Official upstream SHA used by H2.
HUB_REPO = f'facebookresearch/dinov2:{HUB_REF}'
WEIGHTS_SOURCE = 'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth'
FEATURE_DIM = 768
NUM_CLASSES = 20
SEED_AUG = 123
HEAD_SEEDS = [2026]  # Explicit opt-in: [123, 42, 2026]
EXTRACTION_BATCH_SIZE = 8
HEAD_BATCH_SIZE = 64
EPOCHS = 35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
PATIENCE = 7
FORCE_REEXTRACT = False
N_AUG = 4
INCLUDE_ORIGINAL = True
PIPELINE_VERSION = 'I0-v1'
assert HEAD_SEEDS and len(set(HEAD_SEEDS)) == len(HEAD_SEEDS)
assert all(isinstance(seed, int) for seed in HEAD_SEEDS)
assert EXTRACTION_BATCH_SIZE > 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = STORAGE_ROOT / 'student_outputs_I0' / RUN_ID
CACHE_DIR = STORAGE_ROOT / 'student_outputs_I0' / 'cache_vitb14'
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
set_seed(SEED_AUG)
for split in ('train', 'val', 'test'):
    if not (DATA_ROOT / split).is_dir():
        raise FileNotFoundError(f'Missing {DATA_ROOT / split}. Set DATA_ROOT to the extracted course dataset.')
print('Data:', DATA_ROOT, 'Outputs:', OUTPUT_DIR)


## 2. Dataset loading and preprocessing
P3: one deterministic original + four geometric random views; no ColorJitter. Original 700/200 split and 20 prefixed class indices are mandatory.

In [ ]:
# preprocessing change
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=val_transform)
preprocess = val_transform

assert train_ds.class_to_idx == val_ds.class_to_idx, 'train/ and val/ class folders must match'
NUM_CLASSES = len(train_ds.classes)
print('Classes:', NUM_CLASSES, '| Train:', len(train_ds), '| Val:', len(val_ds))
print('First class folders:', train_ds.classes[:5])
assert NUM_CLASSES == 20, f'Expected 20 classes, got {NUM_CLASSES}'

# Class-folder names from the instructor's export are prefixed by 000_, 001_, ...
# These prefixes make ImageFolder indices equal the Kaggle integer labels.
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch at {folder}; ask your instructor'

assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected original 700/200 split.'

## 3. Test image discovery
Require the original sample CSV, preserve its string ID order and never read test labels for training.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
test_dir = DATA_ROOT / 'test'
test_files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
assert test_files, f'No images found inside {test_dir.resolve()}'

sample_path = DATA_ROOT / 'sample_submission.csv'
assert sample_path.exists(), 'Original sample_submission.csv is required.'
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype={'image_id': str})
    assert {'image_id','label'}.issubset(sample.columns)
    required_ids = sample['image_id'].astype(str).tolist()
else:
    required_ids = sorted(p.stem for p in test_files)

by_id = {p.stem: p for p in test_files}
assert len(by_id) == len(test_files), 'Two test filenames share the same image_id stem'
assert len(set(required_ids)) == len(required_ids), 'Duplicate image_id in sample_submission.csv'
assert set(required_ids) == set(by_id), ('Test image IDs do not match sample_submission.csv',
                                         len(set(required_ids)-set(by_id)), len(set(by_id)-set(required_ids)))
print('Unlabeled test images:', len(required_ids))
print('Submission ID example:', required_ids[:3])

class TestImages(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        p = by_id[self.ids[index]]
        with Image.open(p) as im: image = preprocess(im.convert('RGB'))
        return image

test_ds = TestImages(required_ids)
assert len(required_ids) == len(test_files) == len(sample)


## 4. Load frozen DINOv2 ViT-B/14
Official torch.hub entrypoint pinned to the upstream SHA above. Default pretrained=True. No random-weight fallback. Hub caches source and weights. Real output dimensions are checked before any cache can be used.

In [ ]:
set_seed(SEED_AUG)
backbone = torch.hub.load(HUB_REPO, MODEL_NAME, pretrained=True,
                          trust_repo=True, skip_validation=True).to(DEVICE)
backbone.requires_grad_(False)
backbone.eval()
assert backbone.embed_dim == FEATURE_DIM == 768
with torch.inference_mode():
    probe = backbone(torch.zeros(1, 3, 224, 224, device=DEVICE))
assert probe.shape == (1, FEATURE_DIM), probe.shape
assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
FROZEN_PARAMETER_COUNT = sum(p.numel() for p in backbone.parameters())
def weight_digest(model):
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode()); h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_SHA256 = weight_digest(backbone)
del probe
print(MODEL_NAME, FEATURE_DIM, 'frozen parameters:', FROZEN_PARAMETER_COUNT)


## 5. Multi-view feature extraction and caching
View-major layout: original pass first, then four augmented passes; repeat the full label vector five times. Seed once before extraction, never per view. Workers=0, shuffle=False. Content-hashed manifests, actual weight fingerprint, upstream source, transforms and package versions prevent ViT-S/14 or stale reuse. Changing head seeds does not change the cache.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def feature_metadata():
    ordered = ([Path(p) for p, _ in train_ds.samples],
               [Path(p) for p, _ in val_ds.samples], [by_id[i] for i in required_ids])
    manifests = [[{'path': str(p.relative_to(DATA_ROOT)), 'sha256': file_sha256(p)}
                  for p in paths] for paths in ordered]
    sets = [{entry['sha256'] for entry in split} for split in manifests]
    assert not any(sets[i] & sets[j] for i,j in ((0,1),(0,2),(1,2))), 'Duplicate image content across splits'
    return {'pipeline': PIPELINE_VERSION, 'backbone': MODEL_NAME, 'feature_dim': FEATURE_DIM,
            'hub_repo': HUB_REPO, 'weights_source': WEIGHTS_SOURCE, 'weights_sha256': WEIGHTS_SHA256,
            'preprocessing': 'P3_geometry_multiview', 'train_transform': repr(train_transform),
            'val_transform': repr(val_transform), 'seed_aug': SEED_AUG, 'n_aug': N_AUG,
            'include_original': INCLUDE_ORIGINAL, 'layout': 'view-major',
            'class_to_idx': dict(train_ds.class_to_idx), 'manifest': manifests,
            'test_ids': list(required_ids), 'batch_size': EXTRACTION_BATCH_SIZE,
            'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
            'pillow': Image.__version__, 'device': DEVICE}

def metadata_digest(meta):
    return hashlib.sha256(json.dumps(meta, sort_keys=True).encode()).hexdigest()

@torch.inference_mode()
def extract(loader, desc):
    backbone.eval()
    assert not any(p.requires_grad for p in backbone.parameters())
    embeddings = []
    for batch in tqdm(loader, desc=desc):
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        z = backbone(x.to(DEVICE)).float().cpu()
        assert z.ndim == 2 and z.shape[1] == FEATURE_DIM and torch.isfinite(z).all()
        embeddings.append(z)
    return torch.cat(embeddings)

def valid_cache(payload, meta):
    try:
        assert payload['metadata'] == meta
        views = N_AUG + int(INCLUDE_ORIGINAL)
        for key,n in [('X_train',len(train_ds)*views), ('X_val',len(val_ds)), ('X_test',len(required_ids))]:
            x = payload[key]
            assert x.dtype == torch.float32 and x.shape == (n,FEATURE_DIM) and torch.isfinite(x).all()
        assert torch.equal(payload['y_train'], torch.tensor(train_ds.targets).repeat(views))
        assert torch.equal(payload['y_val'], torch.tensor(val_ds.targets))
        return True
    except (KeyError, AssertionError, TypeError, AttributeError):
        return False

def get_features(force=False):
    meta = feature_metadata()
    signature = metadata_digest(meta)
    path = CACHE_DIR / f'I0_dinov2_vitb14_p3_aug{SEED_AUG}_{signature}.pt'
    if path.exists() and not force:
        try:
            payload = torch.load(path, map_location='cpu', weights_only=True)
            if valid_cache(payload, meta):
                print('Loaded compatible cache:', path)
                return payload, signature, path
            print('Incompatible cache; regenerating')
        except Exception as exc:
            print('Unreadable cache; regenerating:', type(exc).__name__)
    set_seed(SEED_AUG)
    kwargs = dict(batch_size=EXTRACTION_BATCH_SIZE, shuffle=False, num_workers=0,
                  pin_memory=(DEVICE == 'cuda'))
    views = []
    original_transform = train_ds.transform
    try:
        if INCLUDE_ORIGINAL:
            train_ds.transform = val_transform
            views.append(extract(DataLoader(train_ds, **kwargs), 'Original training view'))
        train_ds.transform = train_transform
        for index in range(N_AUG):
            views.append(extract(DataLoader(train_ds, **kwargs), f'Augmented training view {index+1}'))
    finally:
        train_ds.transform = original_transform
    payload = {'metadata': meta, 'X_train': torch.cat(views),
               'X_val': extract(DataLoader(val_ds, **kwargs), 'Validation'),
               'X_test': extract(DataLoader(test_ds, **kwargs), 'Test'),
               'y_train': torch.tensor(train_ds.targets).repeat(N_AUG + int(INCLUDE_ORIGINAL)),
               'y_val': torch.tensor(val_ds.targets)}
    assert valid_cache(payload, meta)
    temp = path.with_suffix('.' + uuid.uuid4().hex + '.tmp')
    torch.save(payload, temp); temp.replace(path)
    print('Freshly extracted:', path)
    return payload, signature, path

features, FEATURES_SIGNATURE, CACHE_PATH = get_features(FORCE_REEXTRACT)
X_train, X_val, X_test = [features[k] for k in ('X_train','X_val','X_test')]
y_train, y_val = features['y_train'], features['y_val']
assert X_train.shape == (3500,768) and y_train.shape == (3500,)
assert X_val.shape == (200,768) and X_test.shape == (len(sample),768)
print(X_train.shape, X_val.shape, X_test.shape)


## 6. Feature standardization
Fit to all training views only; save the statistics with each head.

In [ ]:
feature_mean = X_train.mean(dim=0)
feature_std = X_train.std(dim=0, unbiased=False).clamp_min(1e-6)
Z_train = (X_train - feature_mean) / feature_std
Z_val = (X_val - feature_mean) / feature_std
Z_test = (X_test - feature_mean) / feature_std
print('Features standardized without validation/test leakage.')
assert feature_mean.shape == feature_std.shape == (768,)


## 7. Classification-head construction
Exact F0 E4 head, with only its input width changed to 768. No previous classifier weights are loaded.

In [ ]:
HEAD_CONFIG = {'input_dim':768, 'hidden_dim':128, 'use_bn':True, 'dropout':0.3, 'num_classes':20}
def build_head():
    return nn.Sequential(nn.Linear(768,128), nn.BatchNorm1d(128), nn.ReLU(),
                         nn.Dropout(0.3), nn.Linear(128,20))
print(build_head())


## 8. Classifier training
Fresh seeded head, AdamW and independently seeded shuffle loader per run. Accuracy selects the first strictly best epoch, matching F0. Patience=7; at most 35 epochs. No checkpoint or CSV can overwrite another execution’s artifacts.

In [ ]:
def train_classifier(seed):
    # Refuse stale in-memory embeddings after changes to data, transforms or backbone.
    assert weight_digest(backbone) == WEIGHTS_SHA256
    assert FEATURES_SIGNATURE == metadata_digest(feature_metadata())
    assert valid_cache(features, feature_metadata())
    assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
    assert torch.equal(feature_mean, X_train.mean(0))
    assert torch.equal(feature_std, X_train.std(0, unbiased=False).clamp_min(1e-6))
    set_seed(seed)
    head = build_head().to(DEVICE)
    loader = DataLoader(TensorDataset(Z_train,y_train), batch_size=HEAD_BATCH_SIZE,
                        shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed))
    assert len(Z_train) % HEAD_BATCH_SIZE != 1, 'BatchNorm requires more than one training row per batch'
    optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    name = f'I0_dinov2_vitb14_p3_e4_aug{SEED_AUG}_head{seed}'
    checkpoint = OUTPUT_DIR / f'{name}.pt'
    if checkpoint.exists(): raise FileExistsError(f'{checkpoint} exists; rerun from configuration for a new run directory')
    history, best, best_epoch, weights, bad = [], -1., 0, None, 0
    start = time.perf_counter()
    for epoch in range(1,EPOCHS+1):
        head.train(); loss_sum = 0.
        for x,y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(head(x.to(DEVICE)),y.to(DEVICE))
            loss.backward(); optimizer.step()
            loss_sum += loss.item()*len(y)
        head.eval()
        with torch.inference_mode(): pred = head(Z_val.to(DEVICE)).argmax(1).cpu().numpy()
        accuracy = float(accuracy_score(y_val.numpy(),pred))
        f1 = float(f1_score(y_val.numpy(),pred,labels=list(range(20)),average='macro',zero_division=0))
        history.append({'epoch':epoch,'train_loss':loss_sum/len(Z_train),'val_accuracy':accuracy,'macro_f1':f1})
        print(name, epoch, 'accuracy',accuracy,'macro-F1',f1)
        if accuracy > best:
            best, best_epoch, bad = accuracy, epoch, 0
            weights = {k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    head.load_state_dict(weights); head.eval()
    with torch.inference_mode(): val_pred = head(Z_val.to(DEVICE)).argmax(1).cpu().numpy()
    record = {'experiment':'I0','backbone':MODEL_NAME,'seed_aug':SEED_AUG,'seed_head':seed,
              'accuracy':float(accuracy_score(y_val.numpy(),val_pred)),
              'macro_f1':float(f1_score(y_val.numpy(),val_pred,labels=list(range(20)),average='macro',zero_division=0)),
              'correct':int((val_pred==y_val.numpy()).sum()),'val_count':len(y_val),'best_epoch':best_epoch,
              'head_parameters':sum(p.numel() for p in head.parameters() if p.requires_grad),
              'frozen_backbone_parameters':FROZEN_PARAMETER_COUNT,'training_seconds':time.perf_counter()-start,
              'checkpoint':str(checkpoint),'cache_signature':FEATURES_SIGNATURE,'run_name':name}
    payload = {'head_state_dict':weights,'feature_mean':feature_mean,'feature_std':feature_std,
               'class_to_idx':dict(train_ds.class_to_idx),'backbone':MODEL_NAME,'hub_ref':HUB_REF,
               'preprocessing':'P3_geometry_multiview','head_experiment':'E4_mlp_bn_drop03',
               'head_config':HEAD_CONFIG,'seed_aug':SEED_AUG,'seed_head':seed,
               'best_epoch':best_epoch,'best_accuracy':best,'metrics':record,
               'feature_metadata':features['metadata'],'feature_signature':FEATURES_SIGNATURE,
               'training_config':{'epochs':EPOCHS,'batch_size':HEAD_BATCH_SIZE,'learning_rate':LEARNING_RATE,
                                  'weight_decay':WEIGHT_DECAY,'patience':PATIENCE,'optimizer':'AdamW'}}
    with checkpoint.open('xb') as stream: torch.save(payload,stream)
    pd.DataFrame(history).to_csv(OUTPUT_DIR/f'{name}_history.csv',index=False,mode='x')
    (OUTPUT_DIR/f'{name}_metrics.json').write_text(json.dumps(record,indent=2))
    report = classification_report(y_val.numpy(),val_pred,labels=list(range(20)),
                  target_names=train_ds.classes,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(OUTPUT_DIR/f'{name}_classification_report.csv')
    pd.DataFrame(confusion_matrix(y_val.numpy(),val_pred,labels=list(range(20))),
                 index=train_ds.classes,columns=train_ds.classes).to_csv(OUTPUT_DIR/f'{name}_confusion.csv')
    hist = pd.DataFrame(history)
    fig,axes = plt.subplots(1,2,figsize=(10,3))
    axes[0].plot(hist.epoch,hist.train_loss); axes[0].set(ylabel='Train loss',xlabel='Epoch')
    axes[1].plot(hist.epoch,hist.val_accuracy); axes[1].set(ylabel='Validation accuracy',xlabel='Epoch')
    fig.suptitle(name); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f'{name}_curves.png'); plt.close(fig)
    return record

records = [train_classifier(seed) for seed in HEAD_SEEDS]


## 9. Validation evaluation
Metrics below are measured by this execution. Historical F0 values are explicitly marked as reported, not reverified.

In [ ]:
results = pd.DataFrame(records)
display(results[['seed_head','accuracy','macro_f1','correct','val_count','best_epoch','head_parameters','frozen_backbone_parameters']])
results.to_csv(OUTPUT_DIR/'I0_results.csv',index=False)
comparison = [{'model':'DINOv2 ViT-S/14 + E4','seed':2026,'accuracy':0.98,'macro_f1':0.9803,
               'Kaggle':0.94693,'status':'Historical user-reported; not reverified'}]
comparison += [{'model':'DINOv2 ViT-B/14 + E4','seed':r['seed_head'],'accuracy':r['accuracy'],
                'macro_f1':r['macro_f1'],'Kaggle':None,'status':'Measured validation; not submitted'} for r in records]
comparison = pd.DataFrame(comparison)
display(comparison)
comparison.to_csv(OUTPUT_DIR/'I0_F0_comparison.csv',index=False)
if len(records) == 3 and set(results.seed_head) == {123,42,2026}:
    summary = results[['accuracy','macro_f1']].agg(['mean','std'])  # sample SD ddof=1
    display(summary)
    summary.to_csv(OUTPUT_DIR/'I0_multiseed_summary.csv')
else:
    print('Three-seed summary requires explicitly setting HEAD_SEEDS=[123,42,2026].')


## 10. Kaggle submission generation
Restores each selected best checkpoint and its training-derived statistics. Uses no validation labels. Generates CSV only; never submits. Re-running this cell refuses to overwrite CSVs.

In [ ]:
def generate_submission(record):
    checkpoint = torch.load(record['checkpoint'],map_location='cpu',weights_only=True)
    assert checkpoint['backbone'] == MODEL_NAME and checkpoint['feature_signature'] == FEATURES_SIGNATURE
    assert checkpoint['class_to_idx'] == train_ds.class_to_idx
    assert checkpoint['head_config'] == HEAD_CONFIG
    head = build_head().to(DEVICE)
    head.load_state_dict(checkpoint['head_state_dict'],strict=True); head.eval()
    mean, std = checkpoint['feature_mean'],checkpoint['feature_std']
    assert mean.shape == std.shape == (768,) and (std>0).all()
    with torch.inference_mode():
        predictions = head(((X_test-mean)/std).to(DEVICE)).argmax(1).cpu().numpy().astype(int)
    assert len(predictions) == len(sample) == len(required_ids)
    assert ((predictions>=0)&(predictions<20)).all()
    original = pd.read_csv(sample_path,dtype={'image_id':str})
    assert original.image_id.tolist() == required_ids
    submission = pd.DataFrame({'image_id':required_ids,'label':predictions})
    assert submission.image_id.is_unique and list(submission.columns)==['image_id','label']
    path = OUTPUT_DIR / f"I0_dinov2_vitb14_seed{record['seed_head']}_submission.csv"
    submission.to_csv(path,index=False,mode='x')
    return path
submission_paths = [generate_submission(record) for record in records]
print('Generated (not submitted):', submission_paths)


## 11. Experiment summary
Retain the printed run folder (Drive recommended). Report I0 accuracy, macro-F1, correct/200, best epoch and parameter counts; attach I0_results.csv, I0_F0_comparison.csv, metrics JSONs and training curves. For three explicitly configured heads, also report mean ± sample SD. Only record a Kaggle score after a manual submission. No ViT-B/14 results are bundled in this notebook.

In [ ]:
assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
assert weight_digest(backbone) == WEIGHTS_SHA256, 'Frozen backbone changed unexpectedly'
print('Outputs:',OUTPUT_DIR,'\nReusable features:',CACHE_PATH)
display(results)
